某论文的 donchian trend ，实际是依靠长期涨势，效果欠佳。

In [ ]:
import polars as pl
from zh import (
    BTC_USDT,
    CLOSE,
    HIGH,
    LOW,
    OPEN,
    OPEN_TIME,
    SHARPE_MIN,
    SHARPE_DAY,
    UTC,
    read,
)

data = read("1m").lazy()

In [ ]:
window = 10
interval = "1d"
start_date = pl.datetime(2020, 1, 1, time_zone=UTC)
train_date = pl.datetime(2024, 1, 1, time_zone=UTC)
val_date = pl.datetime(2025, 1, 1, time_zone=UTC)
test_date = pl.datetime(2025, 12, 31, time_zone=UTC)

In [ ]:
MAX = CLOSE.rolling_max(window)
MIN = CLOSE.rolling_min(window)
AT_MAX = CLOSE == MAX
AT_MIN = CLOSE == MIN
CLOSE_RET = CLOSE.pct_change()

In [ ]:
data.filter(
    (OPEN_TIME >= start_date) & (OPEN_TIME < train_date), symbol=BTC_USDT
).group_by_dynamic(OPEN_TIME, every=interval).agg(
    OPEN.first(),
    HIGH.max(),
    LOW.min(),
    CLOSE.last(),
).select(
    OPEN_TIME,
    ret=pl.when(AT_MIN.shift(1).or_(AT_MAX.shift(1))).then(CLOSE_RET).otherwise(0),
).select(sharpe=SHARPE_DAY).collect()


In [ ]:
def ts_trend(data, symbol, start_date, end_date, long, short, fees):
    close_ret = CLOSE.pct_change()
    pos = (
        pl.when(long & short)
        .then(0)
        .when(long)
        .then(1)
        .when(short)
        .then(-1)
        .otherwise(None)
    ).forward_fill()
    ret = pos * close_ret - pos.diff().abs() * fees

    return data.filter(
        OPEN_TIME.is_between(start_date, end_date), symbol=symbol
    ).select(OPEN_TIME, ret=ret)

In [ ]:
def donchian_trend(
    max_window: int = 10,
    min_window: int = 10,
    data: pl.LazyFrame = data,
    start_date: pl.Expr = start_date,
    end_date: pl.Expr = train_date,
    symbol: str = BTC_USDT,
    fee_bps: float = 0.0,
) -> pl.LazyFrame:
    max = CLOSE.rolling_max(max_window)
    min = CLOSE.rolling_min(min_window)
    at_max = CLOSE == max
    at_min = CLOSE == min
    fees = fee_bps / 10000
    long = at_max.shift(1)
    short = at_min.shift(1)

    return ts_trend(data, symbol, start_date, end_date, long, short, fees)

In [ ]:
donchian_trend(fee_bps=0.0).select(sharpe=SHARPE_MIN).collect()